In [0]:
%sql
select * from dev.etl.customer_raw

In [0]:
%sql
---- CREATE METADATA SCHEMA
create schema if not exists dev.metadata;

In [0]:
%sql
-- Create Access level table
CREATE TABLE DEV.METADATA.ALLOWED_MKTSEGMENT(
  USER_EMAIL STRING, 
  MKTSEGMENT STRING
)

In [0]:
%sql
SELECT user

In [0]:
%sql

delete from dev.metadata.allowed_mktsegment;
INSERT  INTO DEV.METADATA.ALLOWED_MKTSEGMENT
SELECT current_user() AS user, 'BUILDING' AS mktsegment;
INSERT  INTO DEV.METADATA.ALLOWED_MKTSEGMENT
SELECT 'chaprasi@databricks.com'  AS user, 'MACHINERY' AS mktsegment;

In [0]:
%sql
select * from dev.metadata.allowed_mktsegment

In [0]:
%sql
-- boolean return function to filter data(is_allowed_mktsegment)
CREATE OR REPLACE FUNCTION DEV.metadata.is_allowed_mktsegment(mktsegment STRING)
RETURNS boolean
LANGUAGE SQL
RETURN
exists(
select 1 from dev.metadata.allowed_mktsegment a
where a.mktsegment = is_allowed_mktsegment.mktsegment
and a.user_email = current_user())




In [0]:
%sql
--{NOT EXIST} boolean return function to filter data(is_allowed_mktsegment) with NO EXISTS
CREATE OR REPLACE FUNCTION DEV.metadata.is_allowed_mktsegment(mktsegment STRING)
RETURNS boolean
LANGUAGE SQL
RETURN
not exists(
select 1 from dev.metadata.allowed_mktsegment a
where a.mktsegment = is_allowed_mktsegment.mktsegment
and a.user_email = current_user())


In [0]:
%sql
--- validate
select dev.metadata.is_allowed_mktsegment('MACHINERY'); 

In [0]:
%sql
--- validate
select dev.metadata.is_allowed_mktsegment('BUILDING'); 

In [0]:
%sql
-- DYNAMIC QUERIES
select * from dev.etl.customer_raw where dev.metadata.is_allowed_mktsegment(c_mktsegment);

In [0]:
%sql
-- DYNAMIC VIEW
CREATE OR REPLACE VIEW dev.etl.customer_raw_vw as
select * from dev.etl.customer_raw where dev.metadata.is_allowed_mktsegment(c_mktsegment);

In [0]:
%sql
select * from dev.etl.customer_raw_vw

In [0]:
%sql
--- HOUSEHOLD
select c_mktsegment,count(1) from dev.etl.customer_raw_vw group by c_mktsegment

In [0]:
%sql
-- insert another segment
INSERT  INTO DEV.METADATA.ALLOWED_MKTSEGMENT
SELECT current_user() AS user, 'HOUSEHOLD' AS mktsegment;

In [0]:
%sql
select * from DEV.METADATA.ALLOWED_MKTSEGMENT

In [0]:
%sql
--- RUNNING VALLIDATION QUERY ONCE MORE
select c_mktsegment,count(1) from dev.etl.customer_raw_vw group by c_mktsegment

### APPLY Row level Function for Data Security


In [0]:
%sql
alter table dev.etl.customer_raw set row filter dev.metadata.is_allowed_mktsegment on(c_mktsegment);

In [0]:
%sql describe extended dev.etl.customer_raw 

In [0]:
%sql
select c_mktsegment,count(1) from dev.etl.customer_raw_vw group by c_mktsegment